# CLIP and DINO embeddings

In [ ]:
import os

In [ ]:
base_dir = "C:/Users/lchtu/OneDrive/Desktop/Persoonlijk/werk/RA InfoLAB/Data Transformation/visual-data-transformations/"
image_paths = ["data/FFF_hun/2019-02-26_08-01-54_UTC.jpg","data/FFF_hun/2019-03-02_10-27-43_UTC.jpg", "data/FFF_hun/big_face.jpg","data/FFF_hun/group_marching.jpg","data/FFF_hun/tilted_face.jpg","data/FFF_hun/tilted_up_face.jpg"]
for i,image_file in enumerate(image_paths):
    image_paths[i] = os.path.join(base_dir,image_file)

In [ ]:
from sklearn.decomposition import PCA

def decompose_embeddings_pca(embeddings, n_components=2):
    """
    Reduce high-dimensional embeddings using PCA.

    Args:
        embeddings (numpy.ndarray): Array of embeddings with shape (n_samples, n_features).
        n_components (int, optional): Number of dimensions to retain. Defaults to 2.

    Returns:
        tuple:
            reduced_embeddings:
                PCA-reduced embeddings.

            pca:
                Fitted PCA object.
    """

    pca = PCA(n_components=n_components)
    reduced_embeddings = pca.fit_transform(embeddings)

    return reduced_embeddings, pca

## CLIP

transformer 5.14
sklearn (scikit-learn)

In [ ]:
import torch
from transformers import AutoImageProcessor, CLIPProcessor, CLIPModel
from sklearn.decomposition import PCA
from PIL import Image

c:\Users\lchtu\OneDrive\Desktop\Persoonlijk\werk\RA InfoLAB\Data Transformation\visual-data-transformations\.vtransenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
CLIP_MODEL = "openai/clip-vit-base-patch32"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else "cpu")

In [ ]:
def load_clip_model(clip_model=CLIP_MODEL, device=DEVICE):
    """
    Load a pre-trained CLIP model and processor.

    Args:
        blip_model (str): Hugging Face model identifier for the CLIP model.

    Returns:
        tuple: Loaded CLIP model and processor.
    """
    print("Loading CLIP model...")
    model = CLIPModel.from_pretrained(clip_model)
    model.to(device) 
    model.eval()
    processor = CLIPProcessor.from_pretrained(clip_model)
    return model, processor


def clip_embedding(
    image_file,
    image_processor,
    model,
    normalization,
    device = DEVICE
):
    """
    Generate a CLIP image embedding for an image.

    Args:
        image_file (str): Path to the input image.
        image_processor: Pre-trained CLIP image processor used to prepare the image for the model.
        model: Pre-trained CLIP model used to generate the image embedding.
        normalization (bool): If True the image embedding is normalized 
        device: Device on which the model and input tensors are processed, e.g. "cuda" or "cpu". Defaults "cpu"

    Returns:
        numpy.ndarray: CLIP image embedding as a one-dimensional NumPy array.
    """

    image = Image.open(image_file).convert("RGB")
    inputs = image_processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        vision_outputs = model.vision_model(pixel_values=inputs["pixel_values"])
        embedding = model.visual_projection(vision_outputs.pooler_output)
    if normalization: 
        embedding = embedding / embedding.norm(p=2, dim=-1, keepdim=True)

    return embedding.cpu().numpy()

## DINO

torchvision

In [ ]:
import torch
from transformers import AutoImageProcessor, AutoModel
from sklearn.decomposition import PCA
from PIL import Image

c:\Users\lchtu\OneDrive\Desktop\Persoonlijk\werk\RA InfoLAB\Data Transformation\visual-data-transformations\.vtransenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
DINO_MODEL = 'facebook/dinov2-base'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else "cpu")

In [ ]:
def load_dino_model(dino_model=DINO_MODEL, device=DEVICE):
    """
    Load a pre-trained DINO model and processor.

    Args:
        blip_model (str): Hugging Face model identifier for the DINO model.

    Returns:
        tuple: Loaded DINO model and processor.
    """
    print("Loading DINO model...")
    model = AutoModel.from_pretrained(dino_model)
    model.to(device) 
    model.eval()
    processor = AutoImageProcessor.from_pretrained(dino_model)
    return model, processor


def dino_embedding(
    image_file,
    image_processor,
    model,
    normalization,
    device = DEVICE
):
    """
    Generate a DINO image embedding for an image.

    Args:
        image_file (str): Path to the input image.
        image_processor: Pre-trained DINO image processor used to prepare the image for the model.
        model: Pre-trained DINO model used to generate the image embedding.
        normalization (bool): If True the image embedding is normalized 
        device: Device on which the model and input tensors are processed, e.g. "cuda" or "cpu". Defaults "cpu"

    Returns:
        numpy.ndarray: DINO image embedding as a one-dimensional NumPy array.
    """

    image = Image.open(image_file).convert("RGB")
    inputs = image_processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
    embedding = outputs.pooler_output
    if normalization:
        embedding = embedding / embedding.norm(p=2, dim=-1, keepdim=True)

    return embedding.cpu().numpy()

## BLIP

In [ ]:
import torch
from transformers import Blip2Processor, Blip2Model
from sklearn.decomposition import PCA
from PIL import Image

In [ ]:
BLIP_MODEL = "Salesforce/blip2-opt-2.7b"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else "cpu")

In [ ]:
def load_blip_model(blip_model=BLIP_MODEL, device=DEVICE):
    """
    Load a pre-trained BLIP-2 model and processor.

    Args:
        blip_model (str): Hugging Face model identifier for the BLIP-2 model.

    Returns:
        tuple: Loaded BLIP-2 model and processor.
    """
    print("Loading BLIP model...")
    model = Blip2Model.from_pretrained(blip_model, torch_dtype=torch.float16)
    model.to(device) 
    model.eval()
    processor = Blip2Processor.from_pretrained(blip_model)
    return model, processor


def blip_embedding(
    image_file,
    image_processor,
    model,
    normalization,
    device = DEVICE
):
    """
    Generate a BLIP image embedding for an image.

    Args:
        image_file (str): Path to the input image.
        image_processor: Pre-trained DINO image processor used to prepare the image for the model.
        model: Pre-trained DINO model used to generate the image embedding.
        normalization (bool): If True the image embedding is normalized 
        device: Device on which the model and input tensors are processed, e.g. "cuda" or "cpu". Defaults "cpu"

    Returns:
        numpy.ndarray: DINO image embedding as a one-dimensional NumPy array.
    """

    image = Image.open(image_file).convert("RGB")
    inputs = image_processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        vision_outputs = model.vision_model(pixel_values=inputs["pixel_values"])
    embedding = vision_outputs.pooler_output

    if normalization:
        embedding = embedding / embedding.norm(p=2,dim=-1,keepdim=True)

    return embedding.cpu().numpy()

## Example Usage

In [ ]:
model, processor = load_clip_model(CLIP_MODEL)
embedding = clip_embedding(image_paths[0], processor, model, True, DEVICE)
print(embedding.shape)

Loading CLIP model...


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 4583.95it/s]


(1, 512)


In [ ]:
model, processor = load_dino_model(DINO_MODEL)
embedding = dino_embedding(image_paths[0], processor, model, True, DEVICE)
print(embedding.shape)

Loading DINO model...


Loading weights: 100%|██████████| 223/223 [00:00<00:00, 1467.94it/s]


(1, 768)


In [ ]:
# model, processor = load_blip_model(BLIP_MODEL)
# embedding = blip_embedding(image_paths[0], processor, model, True, DEVICE)
# print(embedding.shape)

Loading BLIP model...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]c:\Users\lchtu\OneDrive\Desktop\Persoonlijk\werk\RA InfoLAB\Data Transformation\visual-data-transformations\.vtransenv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lchtu\.cache\huggingface\hub\models--Salesforce--blip2-opt-2.7b. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-develo

: 